<a href="https://colab.research.google.com/github/fralfaro/MA_M3/blob/main/evaluaciones/tarea_01_manipulacion_kpi_sol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# Tarea 1: Manipulacion de datos y calculo de KPI

## Objetivo

Aplicar tecnicas basicas de manipulacion de datos con Python y Pandas para cargar, limpiar, transformar y resumir un conjunto de datos de ventas e-commerce. Al finalizar, cada grupo debe entregar un notebook reproducible con indicadores comerciales relevantes y conclusiones breves.

## Contexto

Trabajaran con el archivo `evaluaciones/data.csv`, que contiene transacciones de una tienda online. Cada fila representa un producto incluido en una factura.

Columnas principales:

| Columna | Descripcion |
|---|---|
| `InvoiceNo` | Numero de factura. Si comienza con `C`, corresponde a una cancelacion/devolucion. |
| `StockCode` | Codigo del producto. |
| `Description` | Descripcion del producto. |
| `Quantity` | Cantidad vendida o devuelta. |
| `InvoiceDate` | Fecha y hora de la transaccion. |
| `UnitPrice` | Precio unitario. |
| `CustomerID` | Identificador del cliente. |
| `Country` | Pais de la transaccion. |

## Entrega

Entregar este notebook completo, ejecutado y ordenado. Cada respuesta debe incluir codigo, resultado e interpretacion breve.

## 0. Carga de librerias y datos

Cargue el dataset usando `encoding="latin1"`. Luego revise las primeras filas, dimensiones y tipos de datos.

In [4]:
import pandas as pd
import numpy as np

ruta = "https://raw.githubusercontent.com/fralfaro/MA_M3/refs/heads/main/evaluaciones/data.csv"  # Si ejecutas desde la carpeta evaluaciones
df = pd.read_csv(ruta, encoding="latin1")
df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [5]:
df.shape, df.dtypes

((541909, 8),
 InvoiceNo       object
 StockCode       object
 Description     object
 Quantity         int64
 InvoiceDate     object
 UnitPrice      float64
 CustomerID     float64
 Country         object
 dtype: object)

## Preguntas

### 1. Diagnostico inicial de calidad de datos

Responda:

- Cuantas filas y columnas tiene el dataset?
- Que columnas tienen valores nulos?
- Existen facturas duplicadas o filas duplicadas?
- Hay cantidades o precios menores o iguales a cero?
- Cuantas transacciones corresponden a cancelaciones/devoluciones?

In [6]:
diagnostico = pd.DataFrame({
    "metrica": [
        "filas",
        "columnas",
        "filas_duplicadas",
        "facturas_duplicadas",
        "quantity_menor_igual_cero",
        "unitprice_menor_igual_cero",
        "cancelaciones_devoluciones"
    ],
    "valor": [
        df.shape[0],
        df.shape[1],
        df.duplicated().sum(),
        df["InvoiceNo"].duplicated().sum(),
        (df["Quantity"] <= 0).sum(),
        (df["UnitPrice"] <= 0).sum(),
        df["InvoiceNo"].astype(str).str.startswith("C").sum()
    ]
})

nulos = df.isna().sum()
nulos = nulos[nulos > 0]

display(diagnostico)
display(nulos)


,metrica,valor
0,filas,541909
1,columnas,8
2,filas_duplicadas,5268
3,facturas_duplicadas,516009
4,quantity_menor_igual_cero,10624
5,unitprice_menor_igual_cero,2517
6,cancelaciones_devoluciones,9288


,0
Description,1454
CustomerID,135080


### 2. Limpieza y preparacion

Construya un dataframe limpio llamado `df_clean` considerando los siguientes criterios:

- Convertir `InvoiceDate` a formato fecha.
- Crear una variable `is_cancelled` para identificar facturas que comienzan con `C`.
- Eliminar filas duplicadas.
- Excluir cancelaciones/devoluciones para el analisis de ventas.
- Mantener solo registros con `Quantity > 0` y `UnitPrice > 0`.
- Decidir que hacer con los nulos de `CustomerID` y justificar la decision.

Explique brevemente cuantas filas se pierden en el proceso y por que.

In [7]:
filas_iniciales = len(df)

df_clean = df.copy()
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["is_cancelled"] = df_clean["InvoiceNo"].astype(str).str.startswith("C")

df_clean = df_clean.drop_duplicates()
df_clean = df_clean[~df_clean["is_cancelled"]]
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)]
df_clean = df_clean.dropna(subset=["CustomerID"]).copy()

filas_finales = len(df_clean)
filas_perdidas = filas_iniciales - filas_finales

resumen_limpieza = pd.DataFrame({
    "metrica": ["filas_iniciales", "filas_finales", "filas_perdidas"],
    "valor": [filas_iniciales, filas_finales, filas_perdidas]
})

resumen_limpieza


,metrica,valor
0,filas_iniciales,541909
1,filas_finales,392692
2,filas_perdidas,149217


### 3. Variables comerciales

Agregue al dataframe limpio las siguientes variables:

- `TotalAmount`: monto total de la linea (`Quantity * UnitPrice`).
- `Year`, `Month`, `YearMonth`: variables temporales para analisis mensual.
- `InvoiceDateOnly`: fecha sin hora.

Muestre las primeras filas del dataframe resultante.

In [8]:
df_clean["TotalAmount"] = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.month
df_clean["YearMonth"] = df_clean["InvoiceDate"].dt.to_period("M")
df_clean["InvoiceDateOnly"] = df_clean["InvoiceDate"].dt.date

df_clean.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,is_cancelled,TotalAmount,Year,Month,YearMonth,InvoiceDateOnly
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30,2010,12,2010-12,2010-12-01
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34,2010,12,2010-12,2010-12-01
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00,2010,12,2010-12,2010-12-01
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34,2010,12,2010-12,2010-12-01
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34,2010,12,2010-12,2010-12-01


### 4. Calculo de KPI generales

Calcule e interprete los siguientes indicadores:

- Ventas totales.
- Numero de facturas unicas.
- Numero de clientes unicos.
- Unidades vendidas.
- Ticket promedio por factura.
- Venta promedio por cliente.
- Numero de productos unicos vendidos.

Presente los resultados en una tabla resumen.

In [9]:
ventas_totales = df_clean["TotalAmount"].sum()
facturas_unicas = df_clean["InvoiceNo"].nunique()
clientes_unicos = df_clean["CustomerID"].nunique()
unidades_vendidas = df_clean["Quantity"].sum()
ticket_promedio = ventas_totales / facturas_unicas
venta_promedio_cliente = ventas_totales / clientes_unicos
productos_unicos = df_clean["StockCode"].nunique()

kpi_generales = pd.DataFrame({
    "KPI": [
        "Ventas totales",
        "Facturas unicas",
        "Clientes unicos",
        "Unidades vendidas",
        "Ticket promedio por factura",
        "Venta promedio por cliente",
        "Productos unicos vendidos"
    ],
    "Valor": [
        ventas_totales,
        facturas_unicas,
        clientes_unicos,
        unidades_vendidas,
        ticket_promedio,
        venta_promedio_cliente,
        productos_unicos
    ]
})

kpi_generales


,KPI,Valor
0,Ventas totales,8.887209e+06
1,Facturas unicas,1.853200e+04
2,Clientes unicos,4.338000e+03
3,Unidades vendidas,5.152002e+06
4,Ticket promedio por factura,4.795602e+02
5,Venta promedio por cliente,2.048688e+03
6,Productos unicos vendidos,3.665000e+03


### 5. Analisis por pais

Construya una tabla con los 10 paises con mayor venta total. Para cada pais incluya:

- Venta total.
- Cantidad de facturas.
- Clientes unicos.
- Ticket promedio.

Luego responda: que pais concentra mayor venta? Que diferencias observa entre venta total y ticket promedio?

In [10]:
ventas_pais = (
    df_clean
    .groupby("Country")
    .agg(
        Venta_total=("TotalAmount", "sum"),
        Cantidad_facturas=("InvoiceNo", "nunique"),
        Clientes_unicos=("CustomerID", "nunique")
    )
    .reset_index()
)

ventas_pais["Ticket_promedio"] = ventas_pais["Venta_total"] / ventas_pais["Cantidad_facturas"]

top_10_paises = ventas_pais.sort_values("Venta_total", ascending=False).head(10)
top_10_paises


,Country,Venta_total,Cantidad_facturas,Clientes_unicos,Ticket_promedio
35,United Kingdom,7285024.644,16646,3920,437.644157
23,Netherlands,285446.340,94,9,3036.663191
10,EIRE,265262.460,260,3,1020.240231
14,Germany,228678.400,457,94,500.390372
13,France,208934.310,389,87,537.106195
0,Australia,138453.810,57,9,2429.014211
30,Spain,61558.560,90,30,683.984000
32,Switzerland,56443.950,51,21,1106.744118
3,Belgium,41196.340,98,25,420.370816
31,Sweden,38367.830,36,8,1065.773056


### 6. Analisis de productos

Identifique:

- Los 10 productos con mayor venta total.
- Los 10 productos con mayor cantidad vendida.
- Los 10 productos con mayor numero de facturas.

Responda: son los mismos productos en los tres rankings? Que podria explicar las diferencias?

In [11]:
productos = (
    df_clean
    .groupby(["StockCode", "Description"])
    .agg(
        Venta_total=("TotalAmount", "sum"),
        Cantidad_vendida=("Quantity", "sum"),
        Cantidad_facturas=("InvoiceNo", "nunique")
    )
    .reset_index()
)

top_10_productos_venta = productos.sort_values("Venta_total", ascending=False).head(10)
top_10_productos_cantidad = productos.sort_values("Cantidad_vendida", ascending=False).head(10)
top_10_productos_facturas = productos.sort_values("Cantidad_facturas", ascending=False).head(10)

display(top_10_productos_venta)
display(top_10_productos_cantidad)
display(top_10_productos_facturas)


,StockCode,Description,Venta_total,Cantidad_vendida,Cantidad_facturas
2602,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
1318,22423,REGENCY CAKESTAND 3 TIER,142264.75,12374,1703
3459,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100392.10,36706,1971
3444,85099B,JUMBO BAG RED RETROSPOT,85040.54,46078,1600
2100,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916,195
3896,POST,POSTAGE,77803.96,3120,1099
2799,47566,PARTY BUNTING,68785.23,15279,1379
3278,84879,ASSORTED COLOUR BIRD ORNAMENT,56413.03,35263,1375
3894,M,Manual,53419.93,6933,253
2006,23084,RABBIT NIGHT LIGHT,51251.24,27153,801


,StockCode,Description,Venta_total,Cantidad_vendida,Cantidad_facturas
2602,23843,"PAPER CRAFT , LITTLE BIRDIE",168469.60,80995,1
2100,23166,MEDIUM CERAMIC TOP STORAGE JAR,81416.73,77916,195
3020,84077,WORLD WAR 2 GLIDERS ASSTD DESIGNS,13558.41,54319,472
3444,85099B,JUMBO BAG RED RETROSPOT,85040.54,46078,1600
3459,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100392.10,36706,1971
3278,84879,ASSORTED COLOUR BIRD ORNAMENT,56413.03,35263,1375
432,21212,PACK OF 72 RETROSPOT CAKE CASES,16381.88,33670,1029
1108,22197,POPCORN HOLDER,23417.51,30919,632
2006,23084,RABBIT NIGHT LIGHT,51251.24,27153,801
1383,22492,MINI PAINT SET VINTAGE,16039.24,26076,325


,StockCode,Description,Venta_total,Cantidad_vendida,Cantidad_facturas
3459,85123A,WHITE HANGING HEART T-LIGHT HOLDER,100392.10,36706,1971
1318,22423,REGENCY CAKESTAND 3 TIER,142264.75,12374,1703
3444,85099B,JUMBO BAG RED RETROSPOT,85040.54,46078,1600
2799,47566,PARTY BUNTING,68785.23,15279,1379
3278,84879,ASSORTED COLOUR BIRD ORNAMENT,56413.03,35263,1375
174,20725,LUNCH BAG RED RETROSPOT,27868.80,17576,1288
1607,22720,SET OF 3 CAKE TINS PANTRY DESIGN,33298.30,7010,1146
3896,POST,POSTAGE,77803.96,3120,1099
177,20727,LUNCH BAG BLACK SKULL.,17950.30,11246,1052
432,21212,PACK OF 72 RETROSPOT CAKE CASES,16381.88,33670,1029


### 7. Analisis temporal

Calcule la venta mensual y responda:

- Cual fue el mes con mayor venta?
- Cual fue el mes con menor venta?
- Existe algun patron estacional o cambio relevante en el tiempo?

Puede usar tablas en esta tarea; los graficos se evaluan en la Tarea 2.

In [12]:
ventas_mensuales = (
    df_clean
    .groupby("YearMonth")
    .agg(
        Venta_total=("TotalAmount", "sum"),
        Facturas=("InvoiceNo", "nunique"),
        Clientes=("CustomerID", "nunique"),
        Unidades=("Quantity", "sum")
    )
    .reset_index()
)

ventas_mensuales["YearMonth"] = ventas_mensuales["YearMonth"].astype(str)

mes_mayor_venta = ventas_mensuales.sort_values("Venta_total", ascending=False).head(1)
mes_menor_venta = ventas_mensuales.sort_values("Venta_total", ascending=True).head(1)

display(ventas_mensuales)
display(mes_mayor_venta)
display(mes_menor_venta)


,YearMonth,Venta_total,Facturas,Clientes,Unidades
0,2010-12,570422.730,1400,885,311048
1,2011-01,568101.310,987,741,348473
2,2011-02,446084.920,997,758,265027
3,2011-03,594081.760,1321,974,347582
4,2011-04,468374.331,1149,856,291366
5,2011-05,677355.150,1555,1056,372864
6,2011-06,660046.050,1393,991,363014
7,2011-07,598962.901,1331,949,367360
8,2011-08,644051.040,1280,935,397373
9,2011-09,950690.202,1755,1266,543652


,YearMonth,Venta_total,Facturas,Clientes,Unidades
11,2011-11,1156205.61,2657,1664,665923


,YearMonth,Venta_total,Facturas,Clientes,Unidades
2,2011-02,446084.92,997,758,265027


In [13]:
conclusiones = [
    "El pais con mayor venta total es United Kingdom, concentrando la mayor parte de los ingresos.",
    "Se eliminaron registros sin CustomerID para calcular correctamente los indicadores por cliente.",
    "El ticket promedio varia bastante entre paises, lo que indica diferencias en volumen y valor de compra.",
    "Los productos lideres por venta total no siempre coinciden con los productos lideres por cantidad vendida.",
    "Noviembre de 2011 fue el mes con mayor venta, mostrando un aumento relevante hacia fin de anio.",
    "Febrero de 2011 fue el mes con menor venta dentro del periodo analizado.",
    "Se recomienda enfocar acciones comerciales en productos de alta rotacion y paises con alto ticket promedio."
]

for conclusion in conclusiones:
    print("-", conclusion)


- El pais con mayor venta total es United Kingdom, concentrando la mayor parte de los ingresos.
- Se eliminaron registros sin CustomerID para calcular correctamente los indicadores por cliente.
- El ticket promedio varia bastante entre paises, lo que indica diferencias en volumen y valor de compra.
- Los productos lideres por venta total no siempre coinciden con los productos lideres por cantidad vendida.
- Noviembre de 2011 fue el mes con mayor venta, mostrando un aumento relevante hacia fin de anio.
- Febrero de 2011 fue el mes con menor venta dentro del periodo analizado.
- Se recomienda enfocar acciones comerciales en productos de alta rotacion y paises con alto ticket promedio.


## Rubrica sugerida

| Criterio | Ponderacion |
|---|---:|
| Carga, diagnostico y limpieza de datos | 25% |
| Creacion correcta de variables comerciales | 15% |
| Calculo e interpretacion de KPI | 30% |
| Analisis por pais, producto y tiempo | 20% |
| Orden, reproducibilidad y conclusiones | 10% |